# Imports

In [ ]:
import os
import mimetypes
from urllib.parse import urlparse

import pandas as pd
import requests

# Extract Images

In [ ]:
# Paths
csv_path = r"Data\Articles\Articles2024.csv"
output_dir = r"Data\Images\Plain_Images"

# Create output folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Read CSV
df = pd.read_csv(csv_path)

# keep only rows with a non-empty image URL
df = df[df["top_image_url"].notna() & (df["top_image_url"].astype(str).str.strip() != "")]

# Session is a bit faster/more stable than plain requests.get every time
session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

success = 0
failed = 0
skipped = 0

for _, row in df.iterrows():
    article_id = row["article_id"]
    url = str(row["top_image_url"]).strip()

    try:
        # Try to fetch the image
        response = session.get(url, timeout=15, stream=True)
        response.raise_for_status()

        # Check if it is an image
        content_type = response.headers.get("Content-Type", "").lower()
        if not content_type.startswith("image/"):
            print(f"Skipped article {article_id}: URL did not return an image -> {url}")
            skipped += 1
            continue

        # Try to determine extension from content-type
        ext = mimetypes.guess_extension(content_type.split(";")[0]) or ""

        # Fallback: try extension from URL path
        if not ext:
            parsed = urlparse(url)
            _, url_ext = os.path.splitext(parsed.path)
            ext = url_ext if url_ext else ".jpg"

        # Clean odd extensions
        if ext == ".jpe":
            ext = ".jpg"

        filename = f"{article_id}{ext}"
        save_path = os.path.join(output_dir, filename)

        # Save image safely in chunks
        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        print(f"Saved: {filename}")
        success += 1

    except Exception as e:
        # Do not crash if URL is broken / image unavailable
        print(f"Failed article {article_id}: {url} | Error: {e}")
        failed += 1
        continue

print("\nDone.")
print(f"Saved:   {success}")
print(f"Failed:  {failed}")
print(f"Skipped: {skipped}")

Saved: 299560.jpg
Failed article 301048: https://theshiftnews.com/wp-content/uploads/2025/03/Screenshot-2024-02-05-at-7.49.06/AM.png | Error: 404 Client Error: Not Found for url: https://theshiftnews.com/wp-content/uploads/2025/03/Screenshot-2024-02-05-at-7.49.06/AM.png
Saved: 310544.jpg
Saved: 310584.jpg
Saved: 77510.jpg
Saved: 238351.png
Saved: 483029.png
Saved: 80299.jpg
Saved: 310638.jpg
Saved: 218531.jpg
Saved: 485631.png
Saved: 199897.jpg
Saved: 229680.png
Saved: 100661.jpg
Saved: 101554.png
Saved: 104619.jpg
Failed article 388005: https://maltadaily.mt/wp-content/uploads/2024/07/spainweb.jpg | Error: 403 Client Error: Forbidden for url: https://maltadaily.mt/wp-content/uploads/2024/07/spainweb.jpg
Saved: 307281.jpg
Failed article 388592: https://maltadaily.mt/wp-content/uploads/2024/10/stormweb.jpg | Error: 403 Client Error: Forbidden for url: https://maltadaily.mt/wp-content/uploads/2024/10/stormweb.jpg
Saved: 117377.png
Saved: 117418.jpg
Saved: 311044.jpg
Saved: 303952.png
Sav